<div style="border-left:4px solid #36847c;padding:0.4em 1.2em;margin:1em 0"><small>实验笔记　/　01</small><h1>一次异常读数，会把估计拉偏多少？</h1><p>在恒温环境中测量九次。如果一次读数突然偏高，平均值和中位数会怎样变化？</p></div>

**本地 CPU**　·　Python 3 / IPython　·　无需下载数据

选择 **Restart Kernel and Run All Cells** 从头运行，或用 **Shift+Enter** 逐格查看。先看第 3 节的误差表，再把第 1 节的异常幅度改成零，比较两次观察。

## 01　设定比较条件

只改变一次读数的偏差；参考温度、其他读数和两种方法能看到的信息保持一致。参数集中在这里，单位均为摄氏度。

In [ ]:
from random import Random
from statistics import mean, median
from time import perf_counter
from IPython.display import Markdown, display

reference = 20.0
n_readings = 9
offset = 8.0  # 改成 0，可看同一组读数没有人为异常时的结果
seed = 7

## 02　看一眼输入

用固定随机种子生成同一组轻微波动的读数。第二组只把最后一次读数加上指定偏差。两种方法都处理这两组完整数据。

In [ ]:
rng = Random(seed)
clean = [round(reference + rng.uniform(-0.4, 0.4), 2) for _ in range(n_readings)]
changed = clean.copy()
changed[-1] += offset
print("原始读数（°C）：", clean)
print(f"最后一次：{clean[-1]:.2f} → {changed[-1]:.2f} °C")

## 03　哪种估计更接近参考温度？

平均值使用每一个读数；中位数取排在中间的读数。表里保留估计值和**绝对误差**：估计值与参考温度相差几度。先比较同一行条件下的两种方法，再观察加入偏差后各自的误差变化。

In [ ]:
started = perf_counter()
labels = ("原始", "指定偏差")
rows = []
for condition, values in zip(labels, (clean, changed)):
    for name, estimate in (("平均值", mean(values)), ("中位数", median(values))):
        rows.append((condition, name, estimate, abs(estimate - reference)))

table = ["| 条件 | 方法 | 估计值（°C） | 绝对误差（°C）↓ |", "|:---|:---|---:|---:|"]
for condition, name, estimate, error in rows:
    table.append(f"| {condition} | {name} | {estimate:.3f} | {error:.3f} |")
elapsed_s = perf_counter() - started
display(Markdown("\n".join(table)))

## 04　这个结果能说明多少？

下面的观察直接由表格计算。改参数会改变观察；“哪种方法在真实仪器上更好”仍需独立数据验证。

In [ ]:
error_mean, error_median = rows[2][3], rows[3][3]
comparison = ("中位数误差较小" if error_median < error_mean else
              "平均值误差较小" if error_mean < error_median else "两者误差相同")
print(f"实测观察：在指定偏差 {offset:g} °C 的这一组数据上，{comparison}。")
print(f"两种方法的绝对误差相差 {abs(error_mean - error_median):.3f} °C。")
print(f"本次比较与制表计算：{elapsed_s * 1000:.3f} ms（不含启动、输入准备和显示）。")

**解释与边界**　这组数字让我们看到单个偏高读数怎样影响估计。它是已知参考值下的合成示例，仅覆盖当前样本和参数。

**下一步**　如果关心这个观察是否只属于这一组样本，可以先保持条件、更换独立种子。若问题转为真实测量，应使用有可靠参考温度的独立数据。

**复核入口**　第 1 节记录生效参数，第 2 节生成全部输入，第 3 节的 `rows` 保留每一项未舍入的结果。